In [0]:
%pip install osmnx folium
%restart_python

In [0]:
import osmnx as ox
from shapely.geometry import LineString, Point

ox.settings.use_cache = True
ox.settings.cache_folder = "/Volumes/sandbox/danny_schema/dw_volume/osmnx_cache"

def get_route_with_wkt(start_latlng, end_latlng, route_name="Route"):
    """Get route between two points and return WKT data"""
    
    # Download the street network for the area
    G = ox.graph_from_point(start_latlng, dist=20000, network_type="drive") #Limit to 20KM
    
    # Get the nearest network nodes to the start and end points
    orig_node = ox.nearest_nodes(G, start_latlng[1], start_latlng[0])
    dest_node = ox.nearest_nodes(G, end_latlng[1], end_latlng[0])
    
    # Find the shortest path
    route = ox.shortest_path(G, orig_node, dest_node, weight="length")
    
    # Convert to WKT formats
    start_wkt = Point(start_latlng[1], start_latlng[0]).wkt  # longitude, latitude
    end_wkt = Point(end_latlng[1], end_latlng[0]).wkt
    
    # Convert route nodes to LineString WKT
    coords_list = [(G.nodes[node]['x'], G.nodes[node]['y']) for node in route]
    route_linestring = LineString(coords_list)
    route_wkt = route_linestring.wkt
    
    return {
        'route_name': route_name,
        'start_wkt': start_wkt,
        'end_wkt': end_wkt,
        'route_linestring_wkt': route_wkt
    }

# Example usage
start_latlng = (-27.4705, 153.0260)  # Brisbane City
end_latlng = (-28.0167, 153.4000)    # Gold Coast

# Get route data
route_data = get_route_with_wkt(start_latlng, end_latlng, "Brisbane City to Gold Coast")

# Create DataFrame from list of dictionaries
routes_data = [route_data]  # You can add multiple routes here
df = spark.createDataFrame(routes_data)

display(df)

In [0]:
df.createOrReplaceTempView("route")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW route_with_buffer AS
SELECT *, st_astext(ST_Buffer(ST_GeomFromWKT(route_linestring_wkt), 0.001)) AS buffered_route_wkt
FROM route;

SELECT * FROM route_with_buffer

https://www.data.qld.gov.au/dataset/road-condition-roughness-data-and-class-1km-segments/resource/d618ce2e-7d29-4569-97bd-d97bd5831924

1km pavement condition points dataset

In [0]:
%sql
SELECT *, ST_Point(Longitude, Latitude) AS wkt 
FROM sandbox.danny_schema.road_system_condition_roughness_1_km_and_class 
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW joined_results AS
SELECT a.*, b.*
FROM (
  SELECT * , st_astext((ST_Point(Longitude, Latitude))) AS wkt 
  FROM sandbox.danny_schema.road_system_condition_roughness_1_km_and_class
) a
JOIN (
  SELECT *, ST_GeomFromText(buffered_route_wkt) AS route_geom
  FROM route_with_buffer
) b
ON st_contains(b.route_geom, st_geomfromtext(a.wkt));

SELECT * FROM joined_results

In [0]:
import folium
import pandas as pd
from shapely.geometry import Polygon, Point
from shapely import wkt

# Load points in WKT format from the SQL table into a Pandas DataFrame
df_points = spark.sql("SELECT wkt, RoughnessClass FROM joined_results").toPandas()

# Load polygons in WKT format from the SQL table into a Pandas DataFrame
df_polygons = spark.sql("SELECT route_name, buffered_route_wkt FROM joined_results").toPandas()

# Create a color map for different RoughnessClass values
unique_classes = df_points['RoughnessClass'].unique()
color_list = ['red', 'blue', 'green', 'orange', 'purple', 'darkred', 'lightred', 
              'darkblue', 'lightblue', 'darkgreen', 'lightgreen', 'pink', 'gray', 'black']

# Create a dictionary mapping each class to a color
color_map = {cls: color_list[i % len(color_list)] for i, cls in enumerate(unique_classes)}

print("RoughnessClass Color Mapping:")
for cls, color in color_map.items():
    print(f"{cls}: {color}")

# Create a base map
m = folium.Map(location=[-27.4705, 153.0260], zoom_start=10)

# Add polygons to the map
for _, row in df_polygons.iterrows():
    polygon = wkt.loads(row['buffered_route_wkt'])
    geojson = folium.GeoJson(
        data=polygon.__geo_interface__,
        tooltip=row['route_name'],
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 1,
            'fillOpacity': 0.3,
        }
    )
    geojson.add_to(m)

# Add points to the map with colors based on RoughnessClass
for _, row in df_points.iterrows():
    point = wkt.loads(row['wkt'])
    color = color_map.get(row['RoughnessClass'], 'black')  # Default to black if class not in map
    
    folium.CircleMarker(
        location=[point.y, point.x],
        radius=3,  # Increased size for better visibility
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        tooltip=f"Class: {row['RoughnessClass']}"
    ).add_to(m)

# Display the map
display(m)
